# QuRe 3Qformer + caption-conditioned PFM pooling on Kaggle

Train the `3Qformer+pfm` FashionIQ branch on a Kaggle GPU. In Kaggle Notebook settings, enable **GPU** and **Internet** before running the cells. This notebook uses batch size 16, 30 epochs, offline W&B, and saves an epoch checkpoint under `/kaggle/working/QuRe/experiments`.

**FashionIQ Dataset Setup**:
Upload your FashionIQ dataset to Kaggle (via **+ Add Input** -> **Upload Dataset**).
- If uploaded as an uncompressed dataset, it must contain `images/`, `captions/`, and `image_splits/`.
- If uploaded as an archive (`.zip`, `.rar`, `.tar.gz`), the notebook will automatically extract it into `/kaggle/working/fashioniq_extracted`.

In [ ]:
from pathlib import Path
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile
import tarfile

REPO_URL = "https://github.com/fast2104/QuRe.git"
BRANCH = "3Qformer+pfm"
WORK_DIR = Path("/kaggle/working")
REPO_DIR = WORK_DIR / "QuRe"
INPUT_DIR = Path("/kaggle/input")
EXTRACT_DIR = WORK_DIR / "fashioniq_extracted"

BATCH_SIZE = 16
EPOCHS = 30
NUM_WORKERS = 4  # Kaggle is usually more stable with fewer workers than the local server.
PROMPT_FLOW_KL_WEIGHT = 0.01
EXPERIMENT_NAME = "kaggle_3qformer_caption_pool_pfm_bs16"
RESUME_CHECKPOINT = ""  # Optional: full path to a prior caption-pooling run's model.pth in Kaggle Input/Working.

print(f"Python: {sys.version.split()[0]}")
print(f"Kaggle working directory: {WORK_DIR}")

## 1. Import Required Libraries and Setup Repository
Kaggle already provides PyTorch and CUDA. Install QuRe-specific packages with prebuilt wheels to prevent compiling tokenizers from source, then clone the requested branch.

In [ ]:
import sys
import subprocess
import torch

# Install dependencies using --prefer-binary to avoid building wheels from source.
# On Python >= 3.11/3.13, transformers 4.26.1 fails because legacy tokenizers<0.14 lacks wheels.
if sys.version_info < (3, 11):
    %pip install --prefer-binary -q "transformers==4.26.1" "timm==0.4.12" "fairscale==0.4.4" "iopath==0.1.10" "omegaconf==2.3.0" "hydra-core==1.3.2" "ftfy==6.2.3" "openai-clip==1.0.1" "wandb==0.16.3" "webdataset"
else:
    %pip install --prefer-binary -q "transformers>=4.40.0,<5.0.0" "timm>=0.4.12" "fairscale" "iopath" "omegaconf" "hydra-core" "ftfy" "openai-clip" "wandb" "webdataset"

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU accelerator in Kaggle Notebook settings before training.")
print("GPU:", torch.cuda.get_device_name(0))

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    current_branch = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True
    ).strip()
    if current_branch != BRANCH:
        raise RuntimeError(f"Expected branch {BRANCH!r}, found {current_branch!r} in {REPO_DIR}")

os.chdir(REPO_DIR)
print("Repository:", REPO_DIR)
print("Branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())
model_path = REPO_DIR / "lavis/models/blip2_models/blip2_qure.py"
if "self.ref_pool_flow" not in model_path.read_text(encoding="utf-8"):
    raise RuntimeError("This checkout predates caption-conditioned PFM pooling. Start a fresh Kaggle session and run all cells.")
print("QuRe commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 2. Locate and Prepare FashionIQ Dataset
Automatically discovers the FashionIQ dataset in `/kaggle/input`. If the uploaded dataset is in a `.zip` or `.rar` archive, it will be automatically unpacked to `/kaggle/working/fashioniq_extracted`.

In [ ]:
def find_fashioniq_root(*roots):
    for search_root in roots:
        if not search_root.exists():
            continue
        for candidate in [search_root, *search_root.rglob("*")]:
            if not candidate.is_dir():
                continue
            if all((candidate / name).is_dir() for name in ("images", "captions", "image_splits")):
                return candidate
    return None

# 1. First check if dataset is already extracted in /kaggle/input or /kaggle/working
DATASET_ROOT = find_fashioniq_root(INPUT_DIR, EXTRACT_DIR, WORK_DIR)

# 2. If not found, look for archives (.zip, .rar, .tar) in /kaggle/input to extract
if DATASET_ROOT is None and INPUT_DIR.exists():
    archives = [
        path for path in INPUT_DIR.rglob("*")
        if path.is_file() and path.suffix.lower() in {".zip", ".rar", ".tar", ".gz", ".tgz"}
    ]
    if archives:
        EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
        for archive in archives:
            suffix = archive.suffix.lower()
            print(f"Extracting {archive.name} -> {EXTRACT_DIR}")
            if suffix == ".zip":
                with zipfile.ZipFile(archive) as handle:
                    handle.extractall(EXTRACT_DIR)
            elif suffix in {".tar", ".gz", ".tgz"}:
                with tarfile.open(archive, "r:*") as handle:
                    handle.extractall(EXTRACT_DIR)
            elif suffix == ".rar":
                if shutil.which("7z") is None:
                    subprocess.run(["apt-get", "-qq", "update"], check=False)
                    subprocess.run(["apt-get", "-qq", "install", "-y", "p7zip-full"], check=False)
                subprocess.run(
                    ["7z", "x", str(archive), f"-o{EXTRACT_DIR}", "-y"],
                    check=True,
                )
        DATASET_ROOT = find_fashioniq_root(EXTRACT_DIR)

if DATASET_ROOT is None:
    found_inputs = [str(p.relative_to(INPUT_DIR)) for p in INPUT_DIR.glob("*")] if INPUT_DIR.exists() else []
    raise FileNotFoundError(
        "Could not find FashionIQ dataset with required folders ('images', 'captions', 'image_splits').\n"
        f"Contents found in /kaggle/input: {found_inputs or 'None (No dataset attached)'}\n\n"
        "Please ensure you added the dataset in Kaggle:\n"
        "1. In the right sidebar, click '+ Add Input' -> select your uploaded FashionIQ dataset.\n"
        "2. The dataset must contain 'images/', 'captions/', and 'image_splits/' (either directly or inside a .zip/.rar archive)."
    )

print("FashionIQ root:", DATASET_ROOT)
print("Dataset folders:", sorted(path.name for path in DATASET_ROOT.iterdir() if path.is_dir()))

## 3. Load Dataset into DataFrame
FashionIQ annotations are JSON rather than CSV/Excel. Load the training caption pairs into a DataFrame for a quick integrity check before training.

In [ ]:
import pandas as pd

caption_frames = []
for category in ("dress", "shirt", "toptee"):
    caption_file = DATASET_ROOT / "captions" / f"cap.{category}.train.json"
    split_file = DATASET_ROOT / "image_splits" / f"split.{category}.train.json"
    if not caption_file.is_file() or not split_file.is_file():
        raise FileNotFoundError(f"Missing FashionIQ annotation: {caption_file} or {split_file}")
    with caption_file.open(encoding="utf-8") as handle:
        category_rows = pd.DataFrame(json.load(handle))
    category_rows["category"] = category
    caption_frames.append(category_rows)

fashioniq_train_df = pd.concat(caption_frames, ignore_index=True)
print("Training annotation rows:", len(fashioniq_train_df))
fashioniq_train_df.head()

## 4. Explore Dataset Structure
Check annotation fields, category counts, sample captions, and a few referenced images so path or archive problems are caught before the long GPU run.

In [ ]:
print("Shape:", fashioniq_train_df.shape)
print("Columns:", fashioniq_train_df.columns.tolist())
print("Rows by category:")
display(fashioniq_train_df["category"].value_counts())
display(fashioniq_train_df.head(3))

sample_ids = set(fashioniq_train_df.head(20)["candidate"].tolist())
sample_ids.update(fashioniq_train_df.head(20)["target"].tolist())
missing_sample_images = [
    image_id for image_id in sample_ids
    if not (
        (DATASET_ROOT / "images" / f"{image_id}.png").is_file()
        or (DATASET_ROOT / "images" / f"{image_id}.jpg").is_file()
    )
]
print("Images directory files:", sum(1 for path in (DATASET_ROOT / "images").iterdir() if path.is_file()))
if missing_sample_images:
    raise FileNotFoundError(f"Sample image files missing: {missing_sample_images[:5]}")

## 5. Prepare Data for Training
Patch the Kaggle dataset root into the repository's FashionIQ loader, set the requested training values, and validate the model/loss files and all train/validation annotations.

In [ ]:
os.chdir(REPO_DIR)

# 1. Patch base_path in data/fashionIQ.py (idempotent)
loader_path = REPO_DIR / "data" / "fashionIQ.py"
loader_source = loader_path.read_text(encoding="utf-8")
target_base_line = f"base_path = Path({str(DATASET_ROOT)!r})"

if target_base_line in loader_source:
    print("data/fashionIQ.py base_path already correctly configured.")
else:
    loader_source, replacements = re.subn(
        r"^base_path\s*=\s*Path\(.*\)\s*$",
        target_base_line,
        loader_source,
        count=1,
        flags=re.MULTILINE,
    )
    if replacements != 1:
        raise RuntimeError("Could not patch the FashionIQ base_path in data/fashionIQ.py")
    loader_path.write_text(loader_source, encoding="utf-8")
    print(f"data/fashionIQ.py patched with base_path = {DATASET_ROOT}")

# 2. Update experiment training configs
config_path = REPO_DIR / "configs" / "fashionIQ" / "train.json"
train_config = json.loads(config_path.read_text(encoding="utf-8"))
train_config["batch_size"] = BATCH_SIZE
train_config["epoch"] = EPOCHS
train_config["num_workers"] = NUM_WORKERS
train_config["prompt_flow_kl_weight"] = PROMPT_FLOW_KL_WEIGHT
train_config["experiment_description"] = EXPERIMENT_NAME
train_config["eval_frequency"] = 5
config_path.write_text(json.dumps(train_config, indent=2) + "\n", encoding="utf-8")

# 3. Patch train_qure.py with compat polyfill
train_script = REPO_DIR / "train_qure.py"
train_source = train_script.read_text(encoding="utf-8")

compat_header = '''# --- Transformers compatibility polyfill for LAVIS ---
import sys
import types
from importlib.machinery import ModuleSpec
import math
import torch
import torch.nn as nn
import transformers
import transformers.modeling_utils
import transformers.pytorch_utils

# 1. Polyfill model_parallel_utils
if "transformers.utils.model_parallel_utils" not in sys.modules:
    _mp_mod = types.ModuleType("transformers.utils.model_parallel_utils")
    def _assert_device_map(device_map, num_blocks): pass
    def _get_device_map(n_layers, devices):
        layers = list(range(n_layers))
        n_blocks = int(math.ceil(n_layers / len(devices))) if devices else 1
        return {d: layers[i * n_blocks : (i + 1) * n_blocks] for i, d in enumerate(devices)}
    _mp_mod.assert_device_map = _assert_device_map
    _mp_mod.get_device_map = _get_device_map
    sys.modules["transformers.utils.model_parallel_utils"] = _mp_mod

if "transformers.file_utils" not in sys.modules and hasattr(transformers, "utils"):
    sys.modules["transformers.file_utils"] = transformers.utils

# 2. Polyfill optional deps not needed for FashionIQ
for _dummy_pkg in [
    "decord", "webdataset", "open3d", "pycocoevalcap", "pycocoevalcap.eval",
    "pycocotools", "pycocotools.coco", "spacy", "contexttimer",
]:
    if _dummy_pkg not in sys.modules:
        _dmod = types.ModuleType(_dummy_pkg)
        _dmod.__spec__ = ModuleSpec(_dummy_pkg, loader=None)
        _dmod.COCO = lambda *args, **kwargs: None
        _dmod.COCOEvalCap = lambda *args, **kwargs: None
        _dmod.VideoReader = lambda *args, **kwargs: None
        _dmod.cpu = lambda *args, **kwargs: None
        _dmod.gpu = lambda *args, **kwargs: None
        _dmod.bridge = types.ModuleType("bridge")
        _dmod.bridge.set_bridge = lambda *args, **kwargs: None
        _dmod.WebDataset = lambda *args, **kwargs: []
        _dmod.DataPipeline = lambda *args, **kwargs: []
        sys.modules[_dummy_pkg] = _dmod

def _find_pruneable_heads_and_indices(heads, n_heads, head_size, already_pruned_heads):
    mask = torch.ones(n_heads, head_size)
    heads = set(heads) - already_pruned_heads
    for head in heads:
        head = head - sum(1 if h < head else 0 for h in already_pruned_heads)
        mask[head] = 0
    mask = mask.view(-1).contiguous().eq(1)
    index = torch.arange(len(mask))[mask].long()
    return heads, index

def _prune_linear_layer(layer, index, dim=0):
    index = index.to(layer.weight.device)
    W = layer.weight.index_select(dim, index).clone().detach()
    if layer.bias is not None:
        b = layer.bias.clone().detach() if dim == 1 else layer.bias[index].clone().detach()
    new_size = list(layer.weight.size())
    new_size[dim] = len(index)
    new_layer = nn.Linear(new_size[1], new_size[0], bias=layer.bias is not None).to(layer.weight.device)
    new_layer.weight.requires_grad = False
    new_layer.weight.copy_(W)
    new_layer.weight.requires_grad = True
    if layer.bias is not None:
        new_layer.bias.requires_grad = False
        new_layer.bias.copy_(b)
        new_layer.bias.requires_grad = True
    return new_layer

def _apply_chunking_to_forward(forward_fn, chunk_size, chunk_dim, *input_tensors):
    if not input_tensors:
        raise ValueError("Expected at least one input tensor")
    if chunk_size <= 0:
        return forward_fn(*input_tensors)
    length = input_tensors[0].shape[chunk_dim]
    if any(t.shape[chunk_dim] != length for t in input_tensors):
        raise ValueError("Input tensors must match along the chunk dimension")
    if length % chunk_size:
        raise ValueError("Sequence length must be divisible by chunk_size")
    chunks = [t.split(chunk_size, dim=chunk_dim) for t in input_tensors]
    return torch.cat([forward_fn(*parts) for parts in zip(*chunks)], dim=chunk_dim)

for _m in [transformers.modeling_utils, transformers.pytorch_utils, transformers]:
    if not hasattr(_m, "find_pruneable_heads_and_indices"):
        _m.find_pruneable_heads_and_indices = _find_pruneable_heads_and_indices
    if not hasattr(_m, "prune_linear_layer"):
        _m.prune_linear_layer = _prune_linear_layer
    if not hasattr(_m, "apply_chunking_to_forward"):
        _m.apply_chunking_to_forward = getattr(transformers.pytorch_utils, "apply_chunking_to_forward", _apply_chunking_to_forward)

try:
    import transformers.modeling_attn_mask_utils as _mu
    if not hasattr(transformers.modeling_utils, "_expand_mask") and hasattr(_mu, "_expand_mask"):
        transformers.modeling_utils._expand_mask = _mu._expand_mask
except Exception:
    pass

# 3. Polyfill all_tied_weights_keys
def _get_all_tied_keys(self):
    tied = getattr(self, "_tied_weights_keys", {})
    if isinstance(tied, (list, tuple, set)):
        return {k: k for k in tied}
    if isinstance(tied, dict):
        return tied
    return {}

if hasattr(transformers.modeling_utils, "PreTrainedModel"):
    transformers.modeling_utils.PreTrainedModel.all_tied_weights_keys = property(_get_all_tied_keys)
    if hasattr(transformers.modeling_utils.PreTrainedModel, "mark_tied_weights_as_initialized"):
        _orig_mark = transformers.modeling_utils.PreTrainedModel.mark_tied_weights_as_initialized
        def _safe_mark(self, loading_info=None):
            if not hasattr(self, "all_tied_weights_keys"):
                self.all_tied_weights_keys = _get_all_tied_keys(self)
            try:
                return _orig_mark(self, loading_info)
            except Exception:
                pass
        transformers.modeling_utils.PreTrainedModel.mark_tied_weights_as_initialized = _safe_mark

_orig_getattr = nn.Module.__getattr__
def _patched_getattr(self, name):
    if name == "all_tied_weights_keys":
        return _get_all_tied_keys(self)
    return _orig_getattr(self, name)
nn.Module.__getattr__ = _patched_getattr

# 4. Resilient state_dict loading (handles vocab size mismatch 30522 vs 30523)
_orig_load_state_dict = nn.Module.load_state_dict
def _safe_load_state_dict(self, state_dict, strict=True, *args, **kwargs):
    if not strict and isinstance(state_dict, dict):
        model_state = self.state_dict()
        filtered_state_dict = {}
        for k, v in state_dict.items():
            if k in model_state:
                tgt = model_state[k].shape
                if v.shape == tgt:
                    filtered_state_dict[k] = v
                elif len(v.shape) == 1 and len(tgt) == 1:
                    min_len = min(v.shape[0], tgt[0])
                    param = model_state[k].clone()
                    param[:min_len] = v[:min_len]
                    filtered_state_dict[k] = param
                elif len(v.shape) == 2 and len(tgt) == 2 and v.shape[1] == tgt[1]:
                    min_len = min(v.shape[0], tgt[0])
                    param = model_state[k].clone()
                    param[:min_len, :] = v[:min_len, :]
                    filtered_state_dict[k] = param
                else:
                    pass
            else:
                filtered_state_dict[k] = v
        state_dict = filtered_state_dict
    return _orig_load_state_dict(self, state_dict, strict=strict, *args, **kwargs)
nn.Module.load_state_dict = _safe_load_state_dict
# ----------------------------------------------------
'''

train_source = re.sub(
    r"# --- Transformers compatibility (?:patch|polyfill) for LAVIS ---.*?# ----------------------------------------------------\n?",
    "",
    train_source,
    flags=re.DOTALL,
)
train_source = compat_header + train_source
print("train_qure.py patched with full transformers compatibility polyfill.")

train_script.write_text(train_source, encoding="utf-8")

# 4. Patch ONLY specific problematic source files (NOT __init__.py)
# This avoids breaking the registry while still handling optional deps.
_SPECIFIC_PATCHES = {
    # file contains: bad imports  ->  fix
    "lavis/models/blip2_models/blip2.py": [
        # vocab size mismatch: replace strict state_dict load with shape-safe version
        (
            "        msg = self.load_state_dict(state_dict, strict=False)",
            """        model_dict = self.state_dict()
        safe_state_dict = {}
        for _k, _v in state_dict.items():
            if _k in model_dict:
                _tgt = model_dict[_k].shape
                if _v.shape == _tgt:
                    safe_state_dict[_k] = _v
                elif len(_v.shape) == 1 and len(_tgt) == 1:
                    _min = min(_v.shape[0], _tgt[0])
                    _p = model_dict[_k].clone()
                    _p[:_min] = _v[:_min]
                    safe_state_dict[_k] = _p
                elif len(_v.shape) == 2 and len(_tgt) == 2 and _v.shape[1] == _tgt[1]:
                    _min = min(_v.shape[0], _tgt[0])
                    _p = model_dict[_k].clone()
                    _p[:_min, :] = _v[:_min, :]
                    safe_state_dict[_k] = _p
            else:
                safe_state_dict[_k] = _v
        msg = self.load_state_dict(safe_state_dict, strict=False)""",
        ),
    ],
    "lavis/models/blip2_models/Qformer.py": [
        # all_tied_weights_keys class attribute
        (
            "class BertLMHeadModel(BertPreTrainedModel):",
            "class BertLMHeadModel(BertPreTrainedModel):\n    all_tied_weights_keys = {}",
        ),
    ],
    "lavis/models/blip2_models/modeling_t5.py": [
        (
            "from transformers.utils.model_parallel_utils import assert_device_map, get_device_map",
            "try:\n    from transformers.utils.model_parallel_utils import assert_device_map, get_device_map\nexcept ImportError:\n    def assert_device_map(*a, **k): pass\n    def get_device_map(n, d): return {}",
        ),
    ],
    "lavis/runners/runner_iter.py": [
        (
            "from transformers.utils.model_parallel_utils import assert_device_map, get_device_map",
            "try:\n    from transformers.utils.model_parallel_utils import assert_device_map, get_device_map\nexcept ImportError:\n    def assert_device_map(*a, **k): pass\n    def get_device_map(n, d): return {}",
        ),
    ],
    "lavis/runners/runner_base.py": [
        (
            "from transformers.utils.model_parallel_utils import assert_device_map, get_device_map",
            "try:\n    from transformers.utils.model_parallel_utils import assert_device_map, get_device_map\nexcept ImportError:\n    def assert_device_map(*a, **k): pass\n    def get_device_map(n, d): return {}",
        ),
    ],
    "lavis/datasets/data_utils.py": [
        (
            "import decord",
            "try:\n    import decord\nexcept ImportError:\n    decord = None",
        ),
        (
            "import webdataset as wds",
            "try:\n    import webdataset as wds\nexcept ImportError:\n    wds = None",
        ),
    ],
    "lavis/processors/ulip_processors.py": [
        (
            "import open3d",
            "try:\n    import open3d\nexcept ImportError:\n    open3d = None",
        ),
    ],
    "lavis/tasks/captioning.py": [
        (
            "from pycocoevalcap.eval import COCOEvalCap",
            "try:\n    from pycocoevalcap.eval import COCOEvalCap\nexcept ImportError:\n    COCOEvalCap = None",
        ),
    ],
}

for rel_path, patches in _SPECIFIC_PATCHES.items():
    target_file = REPO_DIR / rel_path
    if not target_file.is_file():
        print(f"Skip (not found): {rel_path}")
        continue
    txt = target_file.read_text(encoding="utf-8")
    changed = False
    for old, new in patches:
        if old in txt and new not in txt:
            txt = txt.replace(old, new, 1)
            changed = True
    if changed:
        target_file.write_text(txt, encoding="utf-8")
        print(f"Patched: {rel_path}")
    else:
        print(f"Already OK: {rel_path}")

# 5. Patch __init__.py ONLY for processors/models that import truly optional
#    external packages (open3d, beats, etc.) â€” NOT all from-lavis imports.
#    We identify problematic modules by checking if they themselves import
#    optional deps. Only then do we wrap their import in __init__.py.
_OPTIONAL_MODULES = {
    "lavis.processors.ulip_processors": "open3d",
    "lavis.processors.audio_processors": "beats",
    "lavis.datasets.datasets.laion_dataset": "webdataset",
}

for init_file in (REPO_DIR / "lavis").rglob("__init__.py"):
    if not init_file.is_file():
        continue
    content = init_file.read_text(encoding="utf-8")
    new_lines = []
    changed = False
    for line in content.splitlines():
        if line.startswith("from lavis.") and " import " in line:
            module_part = line.split("from ")[1].split(" import ")[0].strip()
            if module_part in _OPTIONAL_MODULES:
                if f"try:\n    {line}" not in content:
                    new_lines.append(f"try:\n    {line}\nexcept Exception:\n    pass")
                    changed = True
                    continue
        new_lines.append(line)
    if changed:
        init_file.write_text("\n".join(new_lines) + "\n", encoding="utf-8")
        print(f"Selectively patched __init__: {init_file.relative_to(REPO_DIR)}")

# 6. Validate split files
for category in ("dress", "shirt", "toptee"):
    for split in ("train", "val"):
        for folder, filename in (
            ("captions", f"cap.{category}.{split}.json"),
            ("image_splits", f"split.{category}.{split}.json"),
        ):
            if not (DATASET_ROOT / folder / filename).is_file():
                raise FileNotFoundError(DATASET_ROOT / folder / filename)

assert train_config["batch_size"] == 16
assert train_config["epoch"] == 30
assert (REPO_DIR / "lavis" / "models" / "blip2_models" / "prompt_flow.py").is_file()
assert "def contrastive_loss" in train_source
if RESUME_CHECKPOINT:
    RESUME_CHECKPOINT = str(Path(RESUME_CHECKPOINT))
    if not Path(RESUME_CHECKPOINT).is_file():
        raise FileNotFoundError(RESUME_CHECKPOINT)

print(json.dumps({
    "branch": BRANCH,
    "dataset_root": str(DATASET_ROOT),
    "batch_size": train_config["batch_size"],
    "epochs": train_config["epoch"],
    "num_workers": train_config["num_workers"],
    "prompt_flow_kl_weight": train_config["prompt_flow_kl_weight"],
    "eval_frequency": train_config["eval_frequency"],
}, indent=2))

In [ ]:
# Exercise the pooling flow before the full training job.
import runpy
import torch

def check_pool_flow():
    flow_class = runpy.run_path(str(REPO_DIR / "lavis/models/blip2_models/prompt_flow.py"))["PlanarPromptFlow"]
    flow = flow_class(condition_dim=512, hidden_dim=64, num_prompts=32)
    tokens = torch.randn(2, 32, 256)
    caption = torch.randn(2, 256)
    condition = torch.cat([tokens.mean(1), caption], dim=-1)
    logits, kl = flow(condition)
    pooled = (logits.softmax(1).unsqueeze(-1) * tokens).sum(1)
    torch.testing.assert_close(pooled, tokens.mean(1))
    (pooled.square().mean() + 0.01 * kl).backward()
    grad = flow.prompt_decoder[-1].weight.grad
    assert torch.isfinite(kl) and torch.isfinite(grad).all() and grad.abs().max() > 0
    flow.eval()
    with torch.no_grad():
        torch.testing.assert_close(flow(condition)[0], flow(condition)[0])

with torch.random.fork_rng(devices=[]):
    check_pool_flow()
print("PFM pooling check passed.")

## 6. Train QuRe
This trains the branch's three-Q-Former model with its symmetric in-batch contrastive loss plus the configured prompt-flow KL term. Checkpoints are written after every completed epoch. If Kaggle disconnects, set `RESUME_CHECKPOINT` to a saved `model.pth` and rerun from setup.

In [ ]:
import time

os.chdir(REPO_DIR)
os.environ["WANDB_MODE"] = "offline"

train_script = REPO_DIR / "train_qure.py"
if train_script.is_file():
    _src = train_script.read_text(encoding="utf-8")
    if "_safe_load_state_dict" not in _src or "all_tied_weights_keys" not in _src:
        raise RuntimeError("train_qure.py is missing the compatibility patch! Please run Cell 5 first.")

log_path = WORK_DIR / "qure_3qformer_caption_pool_pfm_train.log"
command = [
    sys.executable,
    "-u",
    "train_qure.py",
    "--config_path=configs/fashionIQ/train.json",
    f"--experiment_description={EXPERIMENT_NAME}",
    f"--num_workers={NUM_WORKERS}",
]
if RESUME_CHECKPOINT:
    command.append(f"--checkpoint_path={RESUME_CHECKPOINT}")

print("Running:", " ".join(command))
print("Log:", log_path)
started_at = time.time()
with log_path.open("w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        command,
        cwd=REPO_DIR,
        env=os.environ.copy(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for output_line in process.stdout:
        print(output_line, end="")
        log_file.write(output_line)
        log_file.flush()
    return_code = process.wait()

print(f"Training process exited with code {return_code} after {(time.time() - started_at) / 3600:.2f} hours")
if return_code != 0:
    if log_path.is_file():
        log_lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
        print("\n" + "=" * 30 + " TRAINING LOG (LAST 60 LINES) " + "=" * 30)
        print("\n".join(log_lines[-60:]) if log_lines else "Log file is empty.")
        print("=" * 80 + "\n")
    raise RuntimeError(f"Training exited with code {return_code}; inspect {log_path} and the latest checkpoint.")

## 7. Evaluate Model Performance
The training script runs FashionIQ validation every five epochs and prints retrieval metrics. This cell summarizes completed epochs, the latest saved checkpoint, and validation lines from the log.

In [ ]:
experiment_dirs = sorted(
    (REPO_DIR / "experiments").glob(f"{EXPERIMENT_NAME}_*"),
    key=lambda path: path.stat().st_mtime,
)
if not experiment_dirs:
    raise FileNotFoundError("No QuRe experiment directory was created.")

latest_experiment = experiment_dirs[-1]
checkpoint_path = latest_experiment / "model.pth"
progress_path = latest_experiment / "log.txt"
print("Experiment:", latest_experiment)
print("Checkpoint exists:", checkpoint_path.is_file())
if progress_path.is_file():
    completed_epochs = [
        int(match.group(1))
        for line in progress_path.read_text(encoding="utf-8").splitlines()
        if (match := re.search(r"Epoch\s*:\s*(\d+)", line))
    ]
    print("Last fully completed epoch:", max(completed_epochs) if completed_epochs else "none")
else:
    print("No completed epoch checkpoint log yet.")

if log_path.is_file():
    all_lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
    metric_lines = [
        line for line in all_lines
        if re.search(r"EPOCH|R@|Recall|recall|Validation|FashionIQ", line, re.IGNORECASE)
    ]
    print("\nRecent epoch/validation output:")
    print("\n".join(metric_lines[-80:]) or "No epoch or validation summary found yet.")

print("\nKaggle output files:")
for artifact in (checkpoint_path, latest_experiment / "config.json", progress_path, log_path):
    if artifact.is_file():
        print(f"{artifact} ({artifact.stat().st_size / (1024 ** 3):.2f} GiB) ")


After training, use Kaggle's **Save Version** with notebook outputs enabled. Download `model.pth`, `config.json`, `log.txt`, and `qure_3qformer_caption_pool_pfm_train.log` from the saved version's `/kaggle/working/QuRe/experiments` and `/kaggle/working` outputs. For a later session, add those outputs as a Kaggle Input and set `RESUME_CHECKPOINT` to the prior `model.pth`; keep its sibling `log.txt` beside it so resume starts at the next completed epoch.